# Notebook 1 — The coin and the operator

**Companion to F-004, *Probability as a Time Series*.**

This notebook shows how a fair coin's trajectory, read through the kinematic jet, gives both the classical Binomial *and* the substrate's one-step operator $\hat M$. It is the worked-example version of §§1–5 of the paper. The reader is invited to run each cell and inspect the numbers.


## 1.  A fair coin, $n = 4$

Before any machinery, let's count trajectories.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product

# all 16 trajectories of length 4, encoded as +-1 steps
trajs = list(product([-1, 1], repeat=4))
heads = [sum(1 for s in tr if s == 1) for tr in trajs]
unique = sorted(set(heads))
counts = [heads.count(k) for k in unique]
print('heads count distribution at n=4:')
for k, c in zip(unique, counts):
    print(f'  k={k}:  {c} trajectories  ({c/16:.4f})')
print('row of Pascals triangle:', counts)


The count column `(1, 4, 6, 4, 1)` divided by $2^4 = 16$ is exactly $\mathrm{Bin}(4, \tfrac12)$. The Binomial *is* the count of trajectories.

## 2.  A long fair-coin trajectory

Now generate a long trajectory and look at the running fraction of heads $\hat p_n$.


In [ ]:
rng = np.random.default_rng(42)
n = 5000
steps = rng.choice([-1, 1], size=n)        # +1 = heads, -1 = tails
S = np.cumsum(steps)                        # running net of heads-tails
heads_so_far = np.cumsum(steps == 1)
p_hat = heads_so_far / np.arange(1, n + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
ax1.plot(S[:500], lw=0.8); ax1.set_title('first 500 tosses: cumulative S_n')
ax1.set_xlabel('toss n'); ax1.set_ylabel('heads minus tails'); ax1.grid(alpha=0.3)
ax2.plot(p_hat, color='#3a7ca5'); ax2.axhline(0.5, color='k', lw=0.7, ls='--')
ax2.set_title(r'empirical heads frequency $\hat p_n$')
ax2.set_xlabel('toss n'); ax2.set_ylabel(r'$\hat p_n$'); ax2.set_ylim(0.3, 0.7); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f'after {n} tosses: p_hat = {p_hat[-1]:.4f}')


$\hat p_n$ approaches $1/2$ at rate $1/\sqrt{n}$. The "probability of heads" is what the trajectory *becomes*.

## 3.  The kinematic jet

For the trajectory $Y_t = S_t$ we form the kinematic jet $\mathbf{q}_t = (Y_t, \Delta Y_t, \Delta^2 Y_t)$.


In [ ]:
def kinematic_jet(Y):
    """Kinematic jet using backward differences:
       dY[t] = Y[t] - Y[t-1]   (the increment that led to Y[t])
       ddY[t] = dY[t] - dY[t-1]
    The first two entries are zero-padded for boundary; in practice we
    drop the first 3 samples when fitting."""
    Y = np.asarray(Y, dtype=float)
    dY  = np.zeros_like(Y); dY[1:]  = Y[1:]  - Y[:-1]
    ddY = np.zeros_like(Y); ddY[2:] = dY[2:] - dY[1:-1]
    return np.column_stack([Y, dY, ddY])

Y = S.astype(float)
Q = kinematic_jet(Y)[3:]   # drop boundary
print(f'trajectory length: {len(Y)},  jet length: {len(Q)}')
print('first 10 jets:')
print(np.round(Q[:10], 3))


## 4.  The one-step operator $\hat M$ by OLS

Fit $\mathbf{q}_{t+1} \approx \hat M\, \mathbf{q}_t$ by ordinary least squares.


In [ ]:
def fit_M(Q):
    X, Yt = Q[:-1], Q[1:]
    # ridge for numerical stability
    G = X.T @ X + 1e-8 * np.eye(3)
    return np.linalg.solve(G, X.T @ Yt).T

M_hat = fit_M(Q)
print('M_hat for the fair coin (long trajectory):')
print(np.round(M_hat, 4))
print(f'\nscalar trace s_0 = (1/3) tr(M_hat) = {np.trace(M_hat)/3:.4f}')

# The fitted operator should be the universal iid-trajectory matrix.
# We will compute it once on a long fair coin and use it as the
# empirical "M_iid" reference for the rest of the notebook.
M_iid_empirical = M_hat.copy()


The fitted $\hat M$ for the fair coin is the empirical $\hat M_{\mathrm{iid}}$. The claim of the paper is that this **same** matrix appears for *any* independent-step trajectory — biased coin, Gaussian, Cauchy — regardless of the step's distribution. We verify this in Notebook 2. **The step distribution is not in the operator.**

## 5.  The residual cloud is the step distribution

The residual $\varepsilon_t = \mathbf{q}_{t+1} - \hat M\, \mathbf{q}_t$ carries the part of the next jet the operator cannot predict — i.e., the step itself.


In [ ]:
eps = Q[1:] - Q[:-1] @ M_hat.T
print('residual stats (each column):')
for j, name in enumerate(['eps_Y', 'eps_dY', 'eps_ddY']):
    print(f'  {name}: mean = {eps[:,j].mean(): .4f},  std = {eps[:,j].std():.4f}')

# Show the empirical step distribution lives in the residual
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(eps[:, 1], bins=21, edgecolor='k', alpha=0.7, color='#3a7ca5')
ax.set_title(r'residual cloud, $\varepsilon_{\Delta Y}$ column')
ax.set_xlabel('residual value'); ax.set_ylabel('count'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print('\nThe residual cloud peaks at +-1 — the +-1 steps of the coin.')
print('The marginal of this cloud IS the step distribution, read directly off the trajectory.')


## 6.  A sticky coin: the operator moves

Now suppose the coin is *sticky*: with probability $q = 0.7$, the next toss repeats the last toss; with probability $0.3$, it flips. The marginal is still 50/50 heads, but the trajectory has memory.


In [ ]:
q = 0.7
rng = np.random.default_rng(0)
sticky = np.zeros(n, dtype=int)
sticky[0] = 1
for t in range(1, n):
    if rng.random() < q:
        sticky[t] = sticky[t-1]
    else:
        sticky[t] = -sticky[t-1]

S_st = np.cumsum(sticky).astype(float)
Q_st = kinematic_jet(S_st)[3:]
M_st = fit_M(Q_st)
print('M_hat for the sticky coin:')
print(np.round(M_st, 4))
print(f'\ndisplacement from empirical M_iid: ||M_st - M_iid||_F = {np.linalg.norm(M_st - M_iid_empirical):.4f}')
print(f'scalar trace  s_0 = (1/3) tr(M_st) = {np.trace(M_st)/3:.4f}   (M_iid gives 1/3)')


The trace component $s_0$ moves above $1/3$: the past now predicts the next step, the operator has absorbed the dependence, the residual cloud has shrunk. **Independence is a statement about the operator, not the marginal.**

## 7.  What we showed in seven steps

1. The Binomial is a count of trajectories at horizon $n$.
2. The frequency $\hat p_n$ stabilises along the trajectory; the probability $\tfrac12$ is an asymptote of the trajectory, not a postulate.
3. The kinematic jet $\mathbf{q}_t = (Y_t, \Delta Y_t, \Delta^2 Y_t)$ encodes level, velocity, acceleration of the trajectory.
4. The OLS operator $\hat M$ on the jet lands at $\hat M_{\mathrm{iid}}$ for any independent-step trajectory.
5. The residual cloud reads the step distribution directly off the trajectory.
6. A coin with memory moves the operator away from $\hat M_{\mathrm{iid}}$ in a direction we can name.
7. The whole reading is two coordinates: operator (memory) and residual (step distribution).

This is the trajectory reading of probability, in seven cells.
